# 用 Sciverse 做专利与文献语义探索

> 通过语义检索同时探索专利和学术文献，发现技术关联

**场景**: 研发人员需要了解某项技术在专利和学术论文中的覆盖情况，通过语义检索发现两者之间的关联和差异。

**使用接口**: agentic-search, content

**预估调用量**: ~10–20 次 API 调用

---


## Step 1: 环境准备

安装依赖并配置环境变量


In [ ]:
!pip install httpx anthropic
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # 替换为你的真实值


## Step 2: 语义检索专利和学术文献

分别用专利和学术关键词进行语义检索


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def search(query: str, top_k: int = 15):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        resp.raise_for_status()
        return resp.json()["hits"]

async def main():
    # 检索专利相关内容
    patent_hits = await search("CRISPR base editing patent method composition")
    print(f"Patent-related: {len(patent_hits)} chunks")

    # 检索学术文献
    academic_hits = await search("CRISPR base editing adenine cytosine mechanism")
    print(f"Academic-related: {len(academic_hits)} chunks")

    return patent_hits, academic_hits

patent_hits, academic_hits = asyncio.run(main())


## Step 3: 交叉分析与报告生成

将两组检索结果交给 LLM 进行关联分析


In [ ]:
from anthropic import Anthropic

client = Anthropic()

patent_summary = "\
".join([
    f"- [{h['doc_id']}] (score: {h['score']:.2f}) {h['title']}: {h['chunk'][:80]}..."
    for h in patent_hits[:8]
])
academic_summary = "\
".join([
    f"- [{h['doc_id']}] (score: {h['score']:.2f}) {h['title']}: {h['chunk'][:80]}..."
    for h in academic_hits[:8]
])

msg = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    messages=[{
        "role": "user",
        "content": f"""分析以下两组检索结果的技术关联：

## 专利相关片段
{patent_summary}

## 学术文献片段
{academic_summary}

请输出：
1) 两组结果中的技术主题对比
2) 可能的专利-论文关联（基于内容相似性）
3) 技术发展脉络推测

注意：所有结论必须基于上述检索结果，标注 doc_id。"""
    }]
)
print(msg.content[0].text)


## 注意事项

- 当前为语义探索模式：通过关键词区分专利和学术内容，不保证 100% 准确分类
- 如需精确区分文档类型，请先调用 meta-catalog 确认是否有 source_type 等字段可用
- Sciverse 数据库覆盖学术文献和部分专利，具体覆盖范围请参考数据深度页面
- 建议对关键片段调用 content 接口验证完整上下文后再下结论


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
